In [1]:
# Installation des dépendances
!pip install langchain langchain-openai langchain-community pypdf chromadb python-dotenv -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.

In [2]:
import os

# Configuration clé API sur Google Colab
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("Clé API configurée :", "OK" if os.environ.get("OPENAI_API_KEY") else "MANQUANTE")

Clé API configurée : OK


## Montage Google Drive
Les PDFs du Code du Travail sont stockés sur Google Drive partagé.
Permet à tous d'accéder aux mêmes documents sans avoir à les re-uploader à chaque session Colab.

Chemin utilisé : `Mon Drive/GEN_AI_Assistant_Juridique/data/`

In [6]:
# Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "/content/drive/MyDrive/GEN_AI_Assistant_Juridique/data/LEGITEXT000006072050.pdf"

# Charge uniquement les pages 0 à 299 directement
loader = PyPDFLoader(PDF_PATH)
docs = []
for i, page in enumerate(loader.lazy_load()):
    if i >= 300:
        break
    docs.append(page)

print(f"Pages chargées : {len(docs)}")
print(f"\nAperçu page 1 :")
print(docs[0].page_content[:300])

Pages chargées : 300

Aperçu page 1 :
Code du travail
Partie législative
Chapitre préliminaire : Dialogue social.
Article L1
 
Tout projet de réforme envisagé par le Gouvernement qui porte sur les relations individuelles et collectives
du travail, l'emploi et la formation professionnelle et qui relève du champ de la négociation national


## Découpage en chunks avec RecursiveCharacterTextSplitter

Une fois les documents chargés, on les découpe en chunks

Paramètres choisis :
- `chunk_size = 1000` : chaque chunk fait ~1000 caractères
- `chunk_overlap = 150` : 150 caractères partagés entre deux chunks consécutifs
- `separators` : on découpe en priorité sur les articles de loi ("Article L...")
  pour garder les articles entiers autant que possible

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Découpage en chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\nArticle", "\n\n", "\n", " "]
)

splits = text_splitter.split_documents(docs)

print(f"Nombre de documents originaux : {len(docs)}")
print(f"Nombre de chunks créés : {len(splits)}")
print(f"\nAperçu du premier chunk :")
print(splits[0].page_content[:300])
print(f"\nMétadonnées : {splits[0].metadata}")

Nombre de documents originaux : 300
Nombre de chunks créés : 980

Aperçu du premier chunk :
Code du travail
Partie législative
Chapitre préliminaire : Dialogue social.

Métadonnées : {'producer': 'Apache FOP Version 2.11', 'creator': 'Apache FOP Version 2.11', 'creationdate': '2026-04-09T21:41:42+02:00', 'source': '/content/drive/MyDrive/GEN_AI_Assistant_Juridique/data/LEGITEXT000006072050.pdf', 'total_pages': 3487, 'page': 0, 'page_label': '1'}


## Vectorisation et stockage dans ChromaDB

On utilise le modèle `text-embedding-ada-002` d'OpenAI pour transformer
chaque chunk en vecteur de 1536 dimensions.

ChromaDB est notre base de données vectorielle locale. Elle stocke :
- Les chunks de texte
- Leurs vecteurs associés
- Les métadonnées (source, numéro de page)

Elle permet ensuite de retrouver rapidement les chunks
les plus proches sémantiquement d'une question posée.

In [19]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Chemin de sauvegarde de l'index sur Drive
CHROMA_DIR = "/content/drive/MyDrive/GEN_AI_Assistant_Juridique/chroma_db"

# Création des embeddings et stockage dans ChromaDB
print("Vectorisation en cours... (quelques minutes)")
embedding = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    persist_directory=CHROMA_DIR
)

print(f"✅{len(splits)} chunks vectorisés et stockés dans ChromaDB")
print(f"📁Index sauvegardé dans : {CHROMA_DIR}")

Vectorisation en cours... (quelques minutes)
✅980 chunks vectorisés et stockés dans ChromaDB
📁Index sauvegardé dans : /content/drive/MyDrive/GEN_AI_Assistant_Juridique/chroma_db


## Test du Retrieval

On teste le pipeline en posant une vraie question juridique.
Le retriever va :
1. Transformer la question en vecteur
2. Chercher les chunks les plus proches dans ChromaDB
3. Retourner les k passages les plus pertinents

C'est la validation finale

In [20]:
# Chargement du vectorstore existant
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Test avec une vraie question juridique
question = "Quel est le délai de préavis pour un CDI ?"

print(f"Question : {question}")
print(f"\n{'='*60}")
print("Passages pertinents trouvés dans le Code du Travail :\n")

results = retriever.invoke(question)

for i, doc in enumerate(results):
    print(f"--- Chunk {i+1} (page {doc.metadata.get('page', '?')}) ---")
    print(doc.page_content[:300])
    print()

Question : Quel est le délai de préavis pour un CDI ?

Passages pertinents trouvés dans le Code du Travail :

--- Chunk 1 (page 141) ---
Le préavis ne peut excéder deux semaines.
 
Article L1243-3
 
La rupture anticipée du contrat de travail à durée déterminée qui intervient à l'initiative du salarié en dehors
des cas prévus aux articles L. 1243-1 et L. 1243-2 ouvre droit pour l'employeur à des dommages et intérêts
correspondant au p

--- Chunk 2 (page 110) ---
Article L1234-15
 
Le salarié a droit à un préavis :
 
1° D'un jour lorsque sa rémunération est fixée par jour ;
 
2° D'une semaine lorsque sa rémunération est fixée par semaine ;
 
3° De quinze jours lorsque sa rémunération est fixée par mois ;
 
4° De six semaines lorsque sa rémunération est fixée

--- Chunk 3 (page 107) ---
1° S'il justifie chez le même employeur d'une ancienneté de services continus inférieure à six mois, à un
préavis dont la durée est déterminée par la loi, la convention ou l'accord collectif de travail ou,